# 07 — Experimente nachvollziehbar machen

**Ziel dieses Notebooks** ist, dass jeder Trainingslauf eine Spur hinterlässt:
Parameter, Kennzahlen, Herkunft und das Modell selbst, an einer zentralen Stelle.

**Warum das nötig ist:** Nach dreißig Läufen weiß niemand mehr, welche
Kombination die 0,86 erzeugt hat. Der Zettel mit den Zahlen geht verloren, das
Notebook wird überschrieben, und die Frage „womit war das Modell von letzter
Woche trainiert?" ist nicht mehr beantwortbar.

Bis hierher standen alle Zahlen in Notebooks. Ab jetzt stehen sie in MLflow.

In [1]:
import pandas as pd

import mlflow
from src.modeling.train import EXPERIMENT, TRACKING_URI, trainiere

mlflow.set_tracking_uri(TRACKING_URI)
print("Ablageort der Laufdaten:", TRACKING_URI)
print("Experiment:", EXPERIMENT)

/Users/marzenakanarek/Documents/GitHub/ML Operationalisierung/ST_MLops/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ablageort der Laufdaten: sqlite:////Users/marzenakanarek/Documents/GitHub/ML Operationalisierung/ST_MLops/mlflow.db
Experiment: predictive-maintenance


## 1 Was ein Lauf hinterlässt

Vier Gruppen, und jede beantwortet eine andere Frage.

| Gruppe | beantwortet |
|---|---|
| **Parameter** | Mit welchen Einstellungen wurde gerechnet? |
| **Kennzahlen** | Wie gut war das Ergebnis? |
| **Herkunft** | Welche Daten, welcher Code, welche Bibliotheken? |
| **Artefakte** | Das Modell selbst und die Kennzahlendatei |

Die Herkunftsangaben sind der Teil, den man am leichtesten vergisst und am
dringendsten braucht. Protokolliert werden der SHA-256 der Rohdatei aus
Etappe 3, der Git-Commit, der Zufallszahl-Startwert und die Versionen von
Python, numpy, pandas, scikit-learn und xgboost.

Der Git-Commit bekommt ein `+dirty` angehängt, wenn zum Zeitpunkt des Laufs
nicht festgeschriebene Änderungen im Arbeitsverzeichnis lagen. Ein Lauf mit
`+dirty` ist nicht exakt reproduzierbar — und das soll man sehen.

## 2 Zwei Läufe erzeugen

Damit sich etwas vergleichen lässt, werden zwei Läufe mit unterschiedlichen
Einstellungen ausgeführt: das gewählte Modell und ein bewusst kleineres.

In [2]:
kennzahlen_gewaehlt = trainiere(lauf_name="rf_150_gewaehlt")
print("gewähltes Modell :", {k: round(v, 4) if isinstance(v, float) else v
                             for k, v in kennzahlen_gewaehlt.items()})

kennzahlen_klein = trainiere(
    lauf_name="rf_50_flacher",
    ueberschreibungen={"n_estimators": 50, "max_depth": 6},
)
print("kleineres Modell :", {k: round(v, 4) if isinstance(v, float) else v
                             for k, v in kennzahlen_klein.items()})

2026/10/02 10:49:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'predictive-maintenance' already exists. Creating a new version of this model...
Created version '7' of model 'predictive-maintenance'.
2026/10/02 10:49:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


gewähltes Modell : {'pr_auc': 0.8604, 'roc_auc': 0.952, 'recall': 0.8627, 'precision': 0.5116, 'kosten_eur': 126200.0, 'schwelle': 0.06, 'tp': 44, 'fp': 42, 'fn': 7, 'schranke_bestanden': True, 'verstoesse': []}
kleineres Modell : {'pr_auc': 0.8503, 'roc_auc': 0.9469, 'recall': 0.8431, 'precision': 0.4624, 'kosten_eur': 139400.0, 'schwelle': 0.06, 'tp': 43, 'fp': 50, 'fn': 8, 'schranke_bestanden': True, 'verstoesse': []}


Registered model 'predictive-maintenance' already exists. Creating a new version of this model...
Created version '8' of model 'predictive-maintenance'.


## 3 Die Läufe nebeneinander

Genau das war bisher nicht möglich: zwei Läufe aus verschiedenen Sitzungen in
einer Tabelle vergleichen, ohne beide erneut auszuführen.

In [3]:
laeufe = mlflow.search_runs(experiment_names=[EXPERIMENT])

spalten = {
    "tags.mlflow.runName": "Lauf",
    "params.rf_n_estimators": "Bäume",
    "params.rf_max_depth": "Tiefe",
    "metrics.pr_auc": "PR-AUC",
    "metrics.recall": "Recall",
    "metrics.precision": "Precision",
    "metrics.kosten_eur": "Kosten EUR",
    "params.herkunft_git_commit": "Commit",
}
vorhandene = [s for s in spalten if s in laeufe.columns]
uebersicht = laeufe[vorhandene].rename(columns=spalten)
uebersicht.head(10).round(3)

,Lauf,Bäume,Tiefe,PR-AUC,Recall,Precision,Kosten EUR,Commit
0,rf_50_flacher,50,6,0.850,0.843,0.462,139400.0,2fbf5d8+dirty
1,rf_150_gewaehlt,150,10,0.860,0.863,0.512,126200.0,2fbf5d8+dirty
2,gate_durchgefallen,150,1,0.647,0.451,0.697,303400.0,2fbf5d8+dirty
3,gate_bestanden,150,10,0.860,0.863,0.512,126200.0,2fbf5d8+dirty
4,rf_50_flacher,50,6,0.850,0.843,0.462,139400.0,2fbf5d8+dirty
5,rf_150_gewaehlt,150,10,0.860,0.863,0.512,126200.0,2fbf5d8+dirty
6,gate_durchgefallen,150,1,0.647,0.451,0.697,303400.0,2fbf5d8+dirty
7,gate_bestanden,150,10,0.860,0.863,0.512,126200.0,2fbf5d8+dirty
8,rf_50_flacher,50,6,0.850,0.843,0.462,139400.0,2fbf5d8+dirty
9,rf_150_gewaehlt,150,10,0.860,0.863,0.512,126200.0,2fbf5d8+dirty


Das kleinere Modell verliert rund 0,01 PR-AUC und kostet gut 13.000 € mehr —
eine Information, die man ohne Protokollierung nach dem nächsten Notebook-Lauf
verloren hätte.

Beide Läufe tragen denselben Commit. Würde man jetzt den Code ändern und erneut
trainieren, stünde dort eine andere Kennung — und der Unterschied zwischen den
Läufen wäre dem Code zuzuordnen statt dem Zufall.

In [4]:
# Alles, was zu einem einzelnen Lauf protokolliert wurde.
letzter = mlflow.search_runs(experiment_names=[EXPERIMENT], max_results=1).iloc[0]

print("Herkunft des letzten Laufs:")
for name in sorted(s for s in letzter.index if s.startswith("params.herkunft_")):
    print(f"  {name.replace('params.herkunft_', ''):22s} {letzter[name]}")

print("\nBibliotheksversionen:")
for name in sorted(s for s in letzter.index if s.startswith("params.lib_")):
    print(f"  {name.replace('params.lib_', ''):22s} {letzter[name]}")

Herkunft des letzten Laufs:
  ausfuehrbares_python   /Users/marzenakanarek/Documents/GitHub/ML Operationalisierung/ST_MLops/.venv/bin/python
  git_commit             2fbf5d8+dirty
  plattform              Darwin arm64
  python                 3.13.7
  raw_data_sha256        dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e
  seed                   42

Bibliotheksversionen:
  mlflow                 3.16.1
  numpy                  2.5.3
  pandas                 3.0.6
  python                 3.13.7
  scikit-learn           1.9.1
  xgboost                3.4.1


## 4 Der Abschalter

`trainiere(..., mit_mlflow=False)` beziehungsweise `--no-mlflow` auf der
Kommandozeile führt denselben Lauf aus, hinterlässt aber keine Spur.

Das ist kein Komfort, sondern notwendig: In den Tests aus Etappe 13 und in der
CI aus Etappe 20 würde sonst bei jedem Durchlauf eine Laufdatenbank entstehen.
Testläufe gehören nicht in das Protokoll echter Experimente.

In [5]:
anzahl_vorher = len(mlflow.search_runs(experiment_names=[EXPERIMENT]))
trainiere(mit_mlflow=False)
anzahl_nachher = len(mlflow.search_runs(experiment_names=[EXPERIMENT]))

print(f"Läufe vorher : {anzahl_vorher}")
print(f"Läufe nachher: {anzahl_nachher}")
print("unverändert  :", anzahl_vorher == anzahl_nachher)

Läufe vorher : 10
Läufe nachher: 10
unverändert  : True


## 5 Die Oberfläche

Für den Alltag ist die Tabelle oben praktisch; zum Stöbern und für Abbildungen
ist die mitgelieferte Weboberfläche angenehmer. Im Terminal, aus dem
Projektordner:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Dann `http://localhost:5000` im Browser öffnen. Dort lassen sich Läufe
auswählen, nebeneinanderstellen und ihre Kennzahlen gegen Parameter auftragen.

**Warum SQLite und nicht einfach Dateien:** Das Modellregister in der nächsten
Etappe braucht eine Datenbank. Mit reiner Dateiablage steht es nicht zur
Verfügung. Die Datei `mlflow.db` und der Ordner `mlruns/` sind in `.gitignore`
ausgenommen — Laufdaten sind Arbeitsstände, kein Quelltext.

---

## Zusammenfassung

| | |
|---|---|
| Protokolliert | Hyperparameter, Kennzahlen, Herkunft, Bibliotheksversionen |
| Artefakte | Modell (cloudpickle) und `reports/model_results.json` |
| Ablage | `sqlite:///mlflow.db`, Experiment `predictive-maintenance` |
| Abschaltbar | `--no-mlflow` für Tests und CI |
| Umstellbar | Umgebungsvariable `MLFLOW_TRACKING_URI` |

**Ein Hinweis zur Serialisierung:** MLflow legt sklearn-Modelle neuerdings im
skops-Format ab, das aus Sicherheitsgründen keine selbstgeschriebenen Funktionen
lädt. Unsere Pipeline enthält mit `add_engineered_features` genau so eine
Funktion. Deshalb wird ausdrücklich `cloudpickle` als Format gewählt. Das ist
ein gutes Beispiel dafür, dass eine Pipeline mit eigenem Code zwar das richtige
Vorgehen ist, aber Folgen für die Ablage hat.

Als Nächstes: die Qualitätsschranke. Bisher wird jedes Modell gespeichert — auch
ein schlechteres.